# Few-Shot Learning and In-Context Learning Tutorial

## Overview
This tutorial explores the cutting-edge techniques of Few-Shot Learning and In-Context Learning using **open-source models from HuggingFace** running locally on Google Colab. These methods enable AI models to perform complex tasks with minimal examples, revolutionizing the way we approach machine learning problems.

## Motivation
Traditional machine learning often requires large datasets for training, which can be time-consuming and resource-intensive. Few-Shot Learning and In-Context Learning address this limitation by leveraging the power of large language models to perform tasks with just a handful of examples. This approach is particularly valuable in scenarios where labeled data is scarce or expensive to obtain.

## Key Components
1. **HuggingFace Transformers**: The library used to load and run open-source language models locally.
2. **Qwen3-8B**: An instruction-tuned open-source model that serves as the foundation for our learning techniques.
3. **Chat Templates**: The state-of-the-art approach for structuring few-shot examples as alternating user/assistant messages.
4. **4-bit Quantization (BitsAndBytes)**: Enables running large models efficiently on Colab GPUs.

## Method Details

### 1. Basic Few-Shot Learning
- Implementation of a sentiment classification task using few-shot learning.
- Comparison of two approaches: **chat-template few-shot** (state-of-the-art) vs. **text-embedded few-shot** (traditional).
- Explanation of how the model generalizes from these examples to new inputs.

### 2. Advanced Few-Shot Techniques
- Exploration of multi-task learning for sentiment analysis and language detection.
- Discussion on how to design prompts that enable a single model to perform multiple related tasks.
- Insights into the benefits of this approach, such as improved efficiency and better generalization.

### 3. In-Context Learning
- Demonstration of in-context learning for a custom task (e.g., text transformation).
- Explanation of how models can adapt to new tasks based solely on examples provided in the prompt.
- Discussion on the flexibility and limitations of this approach.

### 4. Best Practices and Evaluation
- Guidelines for selecting effective examples for few-shot learning.
- Techniques for prompt engineering to optimize model performance.
- Implementation of an evaluation framework to assess model accuracy.
- Discussion on the importance of diverse test cases and appropriate metrics.

## Conclusion
Few-Shot Learning and In-Context Learning represent a significant advancement in the field of artificial intelligence. By enabling models to perform complex tasks with minimal examples, these techniques open up new possibilities for AI applications in areas where data is limited. This tutorial provides a solid foundation for understanding and implementing these powerful methods using **open-source models via HuggingFace Transformers**, equipping learners with the tools to leverage large language models effectively in their own projects.

As the field continues to evolve, mastering these techniques will be crucial for AI practitioners looking to stay at the forefront of natural language processing and machine learning.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## The Mechanism of In-Context Learning

Before diving into examples, it's essential to understand **what actually happens** when you give a model a few examples in the prompt.

### Few-Shot Learning Is NOT Fine-Tuning

When you provide examples in a prompt, **the model's weights do not change**. Not a single parameter is updated. This is fundamentally different from fine-tuning, where gradient descent modifies the model's parameters to fit new data.

Instead, something more subtle occurs: the examples you provide create an **implicit task specification** through the model's attention mechanism. Here's how it works:

1. **Pattern Recognition via Attention**: The transformer's self-attention layers identify structural patterns across your examples — specifically the `input → output` mapping that repeats in each example.
2. **Task Inference**: From these patterns, the model internally constructs a representation of "what task is being performed" without any explicit task label.
3. **Pattern Application**: When the model encounters the new input (without an output), it applies the inferred mapping to generate a prediction.

This phenomenon is called **In-Context Learning (ICL)**, and it's an **emergent capability** — it wasn't explicitly trained for, but arises naturally in large language models (typically those with billions of parameters). Smaller models struggle to perform ICL reliably.

### A Mental Model

Think of it like showing someone three solved math problems and then giving them a fourth to solve. You haven't *taught* them math — you've **activated** their existing math knowledge by demonstrating the pattern. The same principle applies to LLMs: few-shot examples don't teach new knowledge, they activate existing capabilities.

In [ ]:
# === Does Example ORDER Matter? ===
# If ICL were just pattern matching, order shouldn't matter.
# But attention has positional encoding — so let's test this.

def classify_with_examples(examples_in_order, test_input):
    """Classify using the given examples in the specified order."""
    messages = [
        {"role": "system", "content": "Classify the text as Positive, Negative, or Neutral. Reply with one word only."},
    ]
    for ex_input, ex_label in examples_in_order:
        messages.append({"role": "user", "content": ex_input})
        messages.append({"role": "assistant", "content": ex_label})
    messages.append({"role": "user", "content": test_input})
    return generate(messages, temperature=0.1).strip()

# Three examples — we'll rearrange these
examples = [
    ("I love this product! It's amazing.", "Positive"),
    ("This is the worst experience I've ever had.", "Negative"),
    ("The meeting is scheduled for 3pm.", "Neutral"),
]

# An ambiguous test case where order might influence the result
ambiguous_input = "I guess the food was fine, nothing special really."

# Test with different orderings
import itertools
orderings = list(itertools.permutations(examples))

print(f"Test input: \"{ambiguous_input}\"\n")
print(f"Testing all {len(orderings)} permutations of 3 examples:\n")

results = {}
for i, ordering in enumerate(orderings):
    last_label = ordering[-1][1]
    result = classify_with_examples(ordering, ambiguous_input)
    results[i] = {"last_example_label": last_label, "prediction": result}
    labels = [ex[1] for ex in ordering]
    order_desc = " -> ".join(labels)
    print(f"Order {i+1}: [{order_desc}] -> Prediction: {result}  (last example was {last_label})")

print("\n--- Summary ---")
unique_predictions = set(r["prediction"] for r in results.values())
print(f"Unique predictions across orderings: {unique_predictions}")
if len(unique_predictions) > 1:
    print("ORDER MATTERS: different orderings produced different predictions!")
else:
    print("All orderings agreed — but this can vary with more ambiguous inputs.")

### Why Does Example Order Matter?

The results above demonstrate that the model's prediction can change depending on example order. This happens because:

1. **Recency Bias in Attention**: Transformer attention patterns show a slight bias toward more recent tokens (those closer to the end of the prompt). The last example often has disproportionate influence on the prediction.
2. **Positional Encoding Effects**: The model's positional encodings mean that tokens at different positions are processed differently, even if their content is identical.
3. **Primacy Effects**: The first example can also carry extra weight because it sets the initial "frame" for the task.

This is strong evidence that the model is performing genuine **attention-based in-context learning**, not simple string matching. If it were just pattern matching on the format, the order wouldn't matter.

> **Practical Tip**: Place your most representative example last, and your second-best example first. Middle examples have the least influence.

## Basic Few-Shot Learning

We'll implement a basic few-shot learning scenario for sentiment classification.

Sentiment Classification:
- Definition: Determining the emotional tone behind a series of words.
- Applications: Customer service, market research, social media analysis.

We compare **two approaches**:

### Approach 1: Chat-Template Few-Shot (State-of-the-Art)
With instruction-tuned models, the best way to provide few-shot examples is as alternating user/assistant messages in the chat template. The model natively understands this turn-based format, leading to more reliable results.

### Approach 2: Text-Embedded Few-Shot (Traditional)
The classic approach embeds all examples into a single text string. This still works but is less natural for chat-tuned models.

In [ ]:
# --- Approach 1: Chat-template few-shot (state-of-the-art) ---
# Examples are provided as alternating user/assistant turns.

def few_shot_sentiment_chat(input_text):
    messages = [
        {"role": "system", "content": "You are a sentiment classifier. Respond with only: Positive, Negative, or Neutral."},
        {"role": "user", "content": "I love this product! It's amazing."},
        {"role": "assistant", "content": "Positive"},
        {"role": "user", "content": "This movie was terrible. I hated it."},
        {"role": "assistant", "content": "Negative"},
        {"role": "user", "content": "The weather today is okay."},
        {"role": "assistant", "content": "Neutral"},
        # Actual query
        {"role": "user", "content": input_text},
    ]
    result = generate(messages, temperature=0.1)
    result = result.strip()
    if ':' in result:
        result = result.split(':')[1].strip()
    return result

test_text = "I can't believe how great this new restaurant is!"
result = few_shot_sentiment_chat(test_text)
print("=== Chat-Template Few-Shot (State-of-the-Art) ===")
print(f"Input: {test_text}")
print(f"Predicted Sentiment: {result}")

In [ ]:
# --- Approach 2: Text-embedded few-shot (traditional) ---
# All examples are embedded in a single prompt string.

def few_shot_sentiment_text(input_text):
    prompt = f"""Classify the sentiment as Positive, Negative, or Neutral.

Examples:
Text: "I love this product! It's amazing." -> Positive
Text: "This movie was terrible. I hated it." -> Negative
Text: "The weather today is okay." -> Neutral

Text: "{input_text}"
Sentiment:"""
    messages = [{"role": "user", "content": prompt}]
    result = generate(messages, temperature=0.1)
    result = result.strip()
    if ':' in result:
        result = result.split(':')[1].strip()
    return result

test_text = "I can't believe how great this new restaurant is!"
result = few_shot_sentiment_text(test_text)
print("=== Text-Embedded Few-Shot (Traditional) ===")
print(f"Input: {test_text}")
print(f"Predicted Sentiment: {result}")

## How Many Examples? The Diminishing Returns Curve

A natural question: if examples help, do more examples always help more? Let's find out empirically.

In [ ]:
# === Testing with 0, 1, 2, 3, and 5 examples ===

# Pool of labeled examples we can draw from
example_pool = [
    ("I absolutely love this! Best purchase ever.", "Positive"),
    ("This is terrible, complete waste of money.", "Negative"),
    ("The package arrived on Tuesday.", "Neutral"),
    ("What a wonderful surprise, I'm so happy!", "Positive"),
    ("I regret buying this, very disappointing.", "Negative"),
]

# Fixed test cases with known labels
test_cases_n = [
    ("This restaurant has amazing food and great service!", "Positive"),
    ("I'm really frustrated with the constant delays.", "Negative"),
    ("The office is located on the 5th floor.", "Neutral"),
    ("What a delightful experience, would recommend!", "Positive"),
    ("Broken on arrival, very poor quality.", "Negative"),
]

def classify_with_n_examples(n, test_input):
    """Classify using the first n examples from the pool."""
    messages = [
        {"role": "system", "content": "Classify the sentiment as Positive, Negative, or Neutral. Reply with one word only."},
    ]
    for ex_input, ex_label in example_pool[:n]:
        messages.append({"role": "user", "content": ex_input})
        messages.append({"role": "assistant", "content": ex_label})
    messages.append({"role": "user", "content": test_input})
    return generate(messages, temperature=0.1).strip()

example_counts = [0, 1, 2, 3, 5]
results_by_count = {}

for n in example_counts:
    correct = 0
    predictions = []
    for test_input, true_label in test_cases_n:
        pred = classify_with_n_examples(n, test_input)
        is_match = pred.lower() == true_label.lower()
        correct += int(is_match)
        predictions.append((test_input[:40], pred, true_label, is_match))
    accuracy = correct / len(test_cases_n)
    results_by_count[n] = {"accuracy": accuracy, "correct": correct, "predictions": predictions}
    sep = "=" * 50
    print(f"\n{sep}")
    print(f" {n}-shot: {correct}/{len(test_cases_n)} correct (accuracy: {accuracy:.0%})")
    print(sep)
    for text, pred, true, ok in predictions:
        mark = "V" if ok else "X"
        print(f"  {mark} \"{text}...\" -> {pred} (expected {true})")

In [ ]:
# === Display results as a summary table ===

header = f"| {'# Examples':^12} | {'Correct':^12} | {'Accuracy':^10} |"
sep_line = f"|{'-'*14}|{'-'*14}|{'-'*12}|"
print(header)
print(sep_line)
for n in example_counts:
    r = results_by_count[n]
    bar = "#" * int(r["accuracy"] * 8)
    n_label = f"{n}-shot"
    correct_label = f'{r["correct"]}/{len(test_cases_n)}'
    acc_label = f'{r["accuracy"]:.0%} {bar}'
    print(f"| {n_label:^12} | {correct_label:^12} | {acc_label:^10} |")

print("\nObservations:")
acc_values = [results_by_count[n]["accuracy"] for n in example_counts]
if acc_values[-1] >= acc_values[-2]:
    print("  - Going from 3 to 5 examples showed minimal or no improvement.")
else:
    print("  - More examples did not always help — possible context dilution.")
if len(acc_values) > 1 and acc_values[0] < acc_values[1]:
    print("  - Even 1 example significantly improved over zero-shot.")
print("  - The biggest jump is typically from 0->1 or 1->2 examples.")

### Why Do Returns Diminish?

The diminishing returns pattern occurs for several interconnected reasons:

1. **Context Token Budget**: Every example consumes tokens from the model's finite context window. More examples = fewer tokens available for the actual task and for the model's "reasoning space."

2. **Attention Dilution**: The attention mechanism must divide its capacity across all tokens in the context. With many examples, the attention weight assigned to any single example decreases, potentially making each individual example's signal weaker.

3. **Pattern Saturation**: The model's internal representation of "what task is this?" typically crystallizes after just 2-3 clear examples. Additional examples confirming the same pattern provide diminishing informational value.

4. **Noise Accumulation**: More examples also mean more opportunities for subtle inconsistencies, unusual phrasing, or edge cases that can confuse the model's task inference.

> **Practical Rule of Thumb**: For most classification tasks, **2-5 examples** hit the sweet spot. Use more only if the task is genuinely ambiguous or has many distinct categories.

## Example Selection Matters

Not all examples are created equal. The *quality* and *diversity* of your examples can matter more than the *quantity*. Let's demonstrate this experimentally.

In [ ]:
# === Diverse Examples vs. Similar Examples ===

# DIVERSE examples: cover different topics, styles, and all three classes
diverse_examples = [
    ("I absolutely adore this new phone, best tech purchase!", "Positive"),
    ("The flight was delayed 6 hours, ruined my whole trip.", "Negative"),
    ("The meeting has been moved to room 204.", "Neutral"),
]

# SIMILAR examples: all about products, all positive, narrow vocabulary
similar_examples = [
    ("Great product, love it!", "Positive"),
    ("Amazing product, so good!", "Positive"),
    ("Wonderful product, highly recommend!", "Positive"),
]

def classify_with_given_examples(examples, test_text):
    messages = [
        {"role": "system", "content": "Classify the sentiment as Positive, Negative, or Neutral. Reply with one word only."},
    ]
    for ex_input, ex_label in examples:
        messages.append({"role": "user", "content": ex_input})
        messages.append({"role": "assistant", "content": ex_label})
    messages.append({"role": "user", "content": test_text})
    return generate(messages, temperature=0.1).strip()

# Test both approaches on multiple inputs
multi_test = [
    ("The customer service was outstanding and really helpful.", "Positive"),
    ("I wasted three hours on hold and got no resolution.", "Negative"),
    ("The building has 12 floors and underground parking.", "Neutral"),
    ("This book changed my perspective on life.", "Positive"),
    ("Completely broken, does not work at all.", "Negative"),
]

print("=== DIVERSE Examples (different topics & all classes) ===")
diverse_correct = 0
for text, label in multi_test:
    pred = classify_with_given_examples(diverse_examples, text)
    ok = pred.lower() == label.lower()
    diverse_correct += int(ok)
    mark = "V" if ok else "X"
    short = text[:50]
    print(f"  {mark} \"{short}...\" -> {pred} (expected {label})")
print(f"  Accuracy: {diverse_correct}/{len(multi_test)}\n")

print("=== SIMILAR Examples (same topic & same class only) ===")
similar_correct = 0
for text, label in multi_test:
    pred = classify_with_given_examples(similar_examples, text)
    ok = pred.lower() == label.lower()
    similar_correct += int(ok)
    mark = "V" if ok else "X"
    short = text[:50]
    print(f"  {mark} \"{short}...\" -> {pred} (expected {label})")
print(f"  Accuracy: {similar_correct}/{len(multi_test)}\n")

print("--- Conclusion ---")
if diverse_correct > similar_correct:
    print(f"Diverse examples ({diverse_correct}/{len(multi_test)}) outperformed similar examples ({similar_correct}/{len(multi_test)}).")
elif diverse_correct == similar_correct:
    print(f"Both scored equally ({diverse_correct}/{len(multi_test)}), but diverse examples are more robust in general.")
else:
    print(f"Similar examples scored higher here, but diverse examples typically generalize better across tasks.")

In [ ]:
# === Adversarial / Contradictory Examples ===
# What happens when we give MISLEADING examples?

# Correct examples (labels match content)
correct_examples = [
    ("I love this! It's wonderful!", "Positive"),
    ("This is awful and terrible.", "Negative"),
    ("The time is 3:30 PM.", "Neutral"),
]

# Contradictory examples (labels CONTRADICT content)
contradictory_examples = [
    ("I love this! It's wonderful!", "Negative"),   # wrong label!
    ("This is awful and terrible.", "Positive"),     # wrong label!
    ("The time is 3:30 PM.", "Neutral"),
]

test_input_adv = "This movie was absolutely fantastic, I loved every minute!"
true_label_adv = "Positive"

print(f"Test: \"{test_input_adv}\"\n")

result_correct = classify_with_given_examples(correct_examples, test_input_adv)
print(f"With CORRECT examples       -> {result_correct}  (expected: {true_label_adv})")

result_adversarial = classify_with_given_examples(contradictory_examples, test_input_adv)
print(f"With CONTRADICTORY examples -> {result_adversarial}  (expected: {true_label_adv})")

print("\n--- Analysis ---")
if result_correct.lower() == true_label_adv.lower() and result_adversarial.lower() != true_label_adv.lower():
    print("The model followed the contradictory examples — proving it genuinely")
    print("uses ICL from examples rather than relying purely on its own knowledge!")
elif result_correct.lower() == true_label_adv.lower() and result_adversarial.lower() == true_label_adv.lower():
    print("The model resisted contradictory examples — its internal knowledge was")
    print("strong enough to override misleading examples for this clear-cut case.")
    print("Try more ambiguous inputs to see stronger adversarial effects.")
else:
    print("Unexpected results. The interplay between ICL and model priors is complex.")

### Understanding Example Selection Through Attention

**Why diverse examples work better:**
- Diverse examples activate **broader relevant patterns** in the model's attention. When examples span different topics and sentiments, the model learns the general task ("classify sentiment") rather than a narrow subtask ("identify positive product reviews").
- With all-positive examples, the model may infer the task is "confirm that text is positive" rather than "classify into three categories."

**Why contradictory examples can mislead:**
- The model faces a conflict between its **pre-trained knowledge** (understanding that "I love this" is positive) and the **in-context signal** (the examples say "I love this" → Negative).
- For clear-cut cases, pre-trained knowledge often wins. But for ambiguous inputs, contradictory examples can easily flip the prediction.
- This tension between model priors and in-context examples is an active area of research in LLM interpretability.

> **Key Insight**: Think of example selection as **programming the attention mechanism**. Each example is an instruction that shapes how the model attends to features of the new input.

## Chat Template vs Text Embedding: Token-Level Comparison

We've seen both approaches work. But what's actually different at the token level? Understanding this reveals why chat-template few-shot is superior for instruction-tuned models.

In [ ]:
# === Approach A: Text-embedded in a single user message ===
text_embedded_prompt = """Classify the sentiment as Positive, Negative, or Neutral.

Input: I love this product! It's amazing.
Output: Positive

Input: This movie was terrible. I hated it.
Output: Negative

Input: The weather today is okay.
Output: Neutral

Input: This restaurant has great food!
Output:"""

text_messages_cmp = [{"role": "user", "content": text_embedded_prompt}]

# === Approach B: Chat-template with user/assistant pairs ===
chat_messages_cmp = [
    {"role": "system", "content": "Classify the sentiment as Positive, Negative, or Neutral. Reply with one word only."},
    {"role": "user", "content": "I love this product! It's amazing."},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "This movie was terrible. I hated it."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "The weather today is okay."},
    {"role": "assistant", "content": "Neutral"},
    {"role": "user", "content": "This restaurant has great food!"},
]

# Tokenize both approaches
text_rendered = tokenizer.apply_chat_template(text_messages_cmp, tokenize=False, add_generation_prompt=True, enable_thinking=False)
chat_rendered = tokenizer.apply_chat_template(chat_messages_cmp, tokenize=False, add_generation_prompt=True, enable_thinking=False)

text_tokens = tokenizer.encode(text_rendered)
chat_tokens = tokenizer.encode(chat_rendered)

sep = "=" * 60
print(sep)
print("APPROACH A: Text-Embedded (all in one user message)")
print(sep)
print(f"Token count: {len(text_tokens)}")
print(f"\nRendered prompt (first 500 chars):\n{text_rendered[:500]}...")

print("\n" + sep)
print("APPROACH B: Chat-Template (user/assistant pairs)")
print(sep)
print(f"Token count: {len(chat_tokens)}")
print(f"\nRendered prompt (first 500 chars):\n{chat_rendered[:500]}...")

print("\n" + sep)
print("COMPARISON")
print(sep)
diff = len(chat_tokens) - len(text_tokens)
more_or_fewer = "more" if diff > 0 else "fewer"
print(f"Chat template uses {abs(diff)} {more_or_fewer} tokens than text-embedded.")
print("The extra tokens come from special role markers (e.g., <|im_start|>, <|im_end|>).")
print("These markers are crucial — they tell the model WHERE each example starts and ends.")

In [ ]:
# === Inspect the special tokens that make chat-template work ===

print("=== Special Tokens in Chat Template ===\n")

# Show the chat-rendered prompt to see structure
lines = chat_rendered.split("\n")
for line in lines:
    has_special = any(tok in line for tok in ["<|im_start|>", "<|im_end|>", "<|endoftext|>"])
    if has_special:
        print(f"  MARKER:  {line}")
    elif line.strip():
        print(f"  CONTENT: {line}")

print("\n--- What the special tokens do ---")
print("  <|im_start|>role  : Signals the START of a message from 'role'")
print("  <|im_end|>        : Signals the END of that message")
print("  These boundaries help the model understand turn structure")
print("  Without them (text-embedded), the model must INFER where examples end")

# Run both approaches on the same input
print("\n=== Output Comparison ===")
result_text = generate(text_messages_cmp, temperature=0.1).strip()
result_chat = generate(chat_messages_cmp, temperature=0.1).strip()
print(f"Text-embedded result: {result_text}")
print(f"Chat-template result: {result_chat}")

### When to Use Each Approach

| Aspect | Chat-Template | Text-Embedded |
|--------|--------------|---------------|
| **Best for** | Instruction-tuned models (Qwen, Llama-chat, etc.) | Base/completion models (GPT-3 base, etc.) |
| **Why it works** | Matches the model's training format exactly | Works as raw text completion |
| **Turn boundaries** | Explicit via special tokens | Must be inferred by the model |
| **Token cost** | Slightly higher (special token overhead) | Slightly lower |
| **Reliability** | Higher — model "knows" each example is separate | Lower — model may blur example boundaries |

**Bottom Line**: For any instruction-tuned model (which is what you'll almost always use in practice), **chat-template few-shot is strictly better**. The small token overhead is a worthwhile trade for explicit turn boundaries that the model was trained to recognize.

The text-embedded approach still has its place: when using base (non-chat) models, when working with APIs that don't support message arrays, or when you need fine-grained control over the exact prompt format.

## Advanced Few-Shot Techniques

We'll now explore multi-task learning for sentiment analysis and language detection.

Multi-task Learning:
- Definition: Training a model to perform multiple related tasks simultaneously.
- Benefits: Improved efficiency, better generalization, reduced overfitting.

Implementation:
1. Design chat messages that include examples for multiple tasks.
2. Use task-specific instructions to guide the model's behavior.
3. Demonstrate how the same model can switch between tasks based on input.

In [ ]:
def multi_task_few_shot(input_text, task):
    messages = [
        {"role": "system", "content": "You perform the requested task on the given text. Reply with only the result, nothing else."},
        {"role": "user", "content": "Task: sentiment\nText: I love this product! It's amazing."},
        {"role": "assistant", "content": "Positive"},
        {"role": "user", "content": "Task: language\nText: Bonjour, comment allez-vous?"},
        {"role": "assistant", "content": "French"},
        # Actual query
        {"role": "user", "content": f"Task: {task}\nText: {input_text}"},
    ]
    return generate(messages, temperature=0.1).strip()

print(multi_task_few_shot("I can't believe how great this is!", "sentiment"))
print(multi_task_few_shot("Guten Tag, wie geht es Ihnen?", "language"))

## In-Context Learning

In-Context Learning allows models to adapt to new tasks based on examples provided in the prompt.

Key Aspects:
1. No fine-tuning required: The model learns from examples in the prompt.
2. Flexibility: Can be applied to a wide range of tasks.
3. Prompt engineering: Careful design of prompts is crucial for performance.

Example Implementation:
We'll demonstrate in-context learning for a custom task (converting text to pig latin) using the `generate()` helper with chat messages.

In [ ]:
def in_context_learning(task_description, examples, input_text):
    messages = [
        {"role": "system", "content": f"You perform the following task: {task_description}. Reply with only the output, nothing else."},
    ]
    for e in examples:
        messages.append({"role": "user", "content": e['input']})
        messages.append({"role": "assistant", "content": e['output']})
    messages.append({"role": "user", "content": input_text})
    return generate(messages, temperature=0.1).strip()

task_desc = "Convert the given text to pig latin."
examples = [
    {"input": "hello", "output": "ellohay"},
    {"input": "apple", "output": "appleay"}
]
test_input = "python"

result = in_context_learning(task_desc, examples, test_input)
print(f"Input: {test_input}")
print(f"Output: {result}")

## When Few-Shot Fails

Few-shot learning is powerful, but it has clear boundaries. Understanding where it fails is just as important as knowing where it succeeds.

In [ ]:
# === Task 1: Complex multi-step reasoning ===
# Few-shot can't teach multi-step logic the model doesn't already handle

reasoning_examples = [
    {"input": "If all roses are flowers and all flowers need water, do roses need water?",
     "output": "Yes: roses are flowers, flowers need water, therefore roses need water."},
    {"input": "If no fish can fly and a salmon is a fish, can a salmon fly?",
     "output": "No: salmon is a fish, no fish can fly, therefore a salmon cannot fly."},
]

# A harder reasoning chain with nonsense words
hard_reasoning = (
    "If all blorks are snazzles, all snazzles are twerps, and all twerps can glimmer, "
    "can a blork glimmer?"
)

messages_reason = [
    {"role": "system", "content": "Answer the logic question with a step-by-step explanation. Be precise."},
]
for ex in reasoning_examples:
    messages_reason.append({"role": "user", "content": ex["input"]})
    messages_reason.append({"role": "assistant", "content": ex["output"]})
messages_reason.append({"role": "user", "content": hard_reasoning})

result_reason = generate(messages_reason, temperature=0.1, max_new_tokens=256)
print("=== Complex Reasoning with Nonsense Words ===")
print(f"Question: {hard_reasoning}\n")
print(f"Model answer: {result_reason.strip()}")
print("\nExpected: Yes — blork -> snazzle -> twerp -> can glimmer")

# === Task 2: Domain-specific knowledge the model doesn't have ===
sep = "=" * 50
print(f"\n{sep}")
print("=== Domain-Specific Knowledge ===")

domain_examples = [
    {"input": "What is the melting point of iron?", "output": "1,538 C"},
    {"input": "What is the melting point of gold?", "output": "1,064 C"},
    {"input": "What is the melting point of copper?", "output": "1,085 C"},
]

# Ask about a fictional compound
domain_query = "What is the melting point of Compound XR-7742?"

messages_domain = [
    {"role": "system", "content": "Answer with just the temperature value."},
]
for ex in domain_examples:
    messages_domain.append({"role": "user", "content": ex["input"]})
    messages_domain.append({"role": "assistant", "content": ex["output"]})
messages_domain.append({"role": "user", "content": domain_query})

result_domain = generate(messages_domain, temperature=0.1, max_new_tokens=64)
print(f"Question: {domain_query}")
print(f"Model answer: {result_domain.strip()}")
print("Note: 'Compound XR-7742' is fictional — the model may hallucinate a confident answer!")

In [ ]:
# === Confusing examples that HURT performance ===
# When examples conflict with the task or with each other

print("=== Impact of Confusing vs. Clear Examples ===\n")

test_inputs_fail = [
    ("The sunset was absolutely beautiful tonight.", "Positive"),
    ("My computer crashed and I lost all my work.", "Negative"),
    ("Water boils at 100 degrees Celsius.", "Neutral"),
]

# CLEAR examples: consistent format, correct labels, unambiguous
clear_examples = [
    ("I'm thrilled with my new car!", "Positive"),
    ("The service was horrendous and rude.", "Negative"),
    ("The store opens at 9 AM.", "Neutral"),
]

# CONFUSING examples: ambiguous text, mixed signals
confusing_examples = [
    ("It's not bad but it's not great either, I suppose it's okay maybe", "Positive"),
    ("I didn't not dislike it, well sort of, it wasn't the worst", "Negative"),
    ("Honestly I have mixed feelings, leaning slightly positively perhaps", "Neutral"),
]

print("--- With CLEAR examples ---")
clear_correct = 0
for text, label in test_inputs_fail:
    pred = classify_with_given_examples(clear_examples, text)
    ok = pred.lower() == label.lower()
    clear_correct += int(ok)
    mark = "V" if ok else "X"
    short = text[:45]
    print(f"  {mark} \"{short}...\" -> {pred} (expected {label})")
print(f"  Accuracy: {clear_correct}/{len(test_inputs_fail)}\n")

print("--- With CONFUSING examples ---")
confuse_correct = 0
for text, label in test_inputs_fail:
    pred = classify_with_given_examples(confusing_examples, text)
    ok = pred.lower() == label.lower()
    confuse_correct += int(ok)
    mark = "V" if ok else "X"
    short = text[:45]
    print(f"  {mark} \"{short}...\" -> {pred} (expected {label})")
print(f"  Accuracy: {confuse_correct}/{len(test_inputs_fail)}")

print("\n--- Takeaway ---")
if clear_correct >= confuse_correct:
    print("Clear, unambiguous examples led to better or equal performance.")
print("Confusing examples make the model uncertain about what the task even is.")

### The Boundary of Few-Shot Learning

These experiments reveal a fundamental principle:

**Few-shot learning can only *activate* patterns the model already knows — it cannot *teach* new knowledge.**

Specifically, few-shot fails or degrades when:

1. **The task requires knowledge the model doesn't have**: No number of examples will teach a model facts it never learned during pre-training. The model may hallucinate a confident but wrong answer instead.

2. **The reasoning chain is too long**: While examples can demonstrate the *format* of step-by-step reasoning, they can't fundamentally improve the model's logical depth. A model that struggles with 4-step reasoning won't suddenly handle it after seeing 2-step examples.

3. **Examples are ambiguous or contradictory**: If the examples themselves are unclear, the model can't infer a clean task specification. Garbage in, garbage out.

4. **The task requires counting, exact math, or structured data manipulation**: LLMs are fundamentally next-token predictors, not calculators. Few-shot examples of arithmetic won't make the model better at arithmetic.

> **When few-shot isn't enough**, consider: (a) fine-tuning with actual training data, (b) retrieval-augmented generation (RAG) for knowledge tasks, (c) chain-of-thought prompting for reasoning tasks, or (d) tool use (letting the model call external calculators, databases, etc.).

## Best Practices and Evaluation

To maximize the effectiveness of few-shot and in-context learning:

1. Example Selection:
   - Diversity: Cover different aspects of the task.
   - Clarity: Use unambiguous examples.
   - Relevance: Choose examples similar to expected inputs.
   - Balance: Ensure equal representation of classes/categories.
   - Edge cases: Include examples of unusual or difficult cases.

2. Prompt Engineering:
   - Clear instructions: Specify the task explicitly.
   - Consistent format: Maintain a uniform structure for examples and inputs.
   - Conciseness: Avoid unnecessary information that may confuse the model.
   - For instruction-tuned models, prefer chat-template few-shot over text-embedded.

3. Evaluation:
   - Create a diverse test set.
   - Compare model predictions to true labels.
   - Use appropriate metrics (e.g., accuracy, F1 score) based on the task.

Below we evaluate **both** few-shot approaches (chat-template and text-embedded) on the same test set.

In [ ]:
def evaluate_model(model_func, test_cases):
    '''
    Evaluate the model on a set of test cases.

    Args:
    model_func: The function that makes predictions.
    test_cases: A list of dicts with "input" and "label" keys.

    Returns:
    The accuracy of the model on the test cases.
    '''
    correct = 0
    total = len(test_cases)

    for case in test_cases:
        input_text = case['input']
        true_label = case['label']
        prediction = model_func(input_text).strip()

        is_correct = prediction.lower() == true_label.lower()
        correct += int(is_correct)

        print(f"Input: {input_text}")
        print(f"Predicted: {prediction}")
        print(f"Actual: {true_label}")
        print(f"Correct: {is_correct}\n")

    accuracy = correct / total
    return accuracy

test_cases = [
    {"input": "This product exceeded my expectations!", "label": "Positive"},
    {"input": "I'm utterly disappointed with the service.", "label": "Negative"},
    {"input": "The temperature today is 72 degrees.", "label": "Neutral"}
]

print("=" * 50)
print("Chat-Template Few-Shot (State-of-the-Art)")
print("=" * 50)
acc_chat = evaluate_model(few_shot_sentiment_chat, test_cases)
print(f"Accuracy: {acc_chat:.2f}\n")

print("=" * 50)
print("Text-Embedded Few-Shot (Traditional)")
print("=" * 50)
acc_text = evaluate_model(few_shot_sentiment_text, test_cases)
print(f"Accuracy: {acc_text:.2f}")